In [3]:
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.decomposition import LatentDirichletAllocation


In [4]:
docs = [

    "Machine learning is a branch of artificial intelligence that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention.",
    
    "Deep learning is a branch of machine learning.",
    
    "Quantum computers use quantum bits, or qubits, which can exist in multiple states simultaneously, unlike traditional computers that use bits.",
]

In [5]:
# Convert to bag-of-words

vectorizer = CountVectorizer(stop_words='english')

X = vectorizer.fit_transform(docs)

X

<3x28 sparse matrix of type '<class 'numpy.int64'>'
	with 31 stored elements in Compressed Sparse Row format>

In [6]:
# Train LDA

lda = LatentDirichletAllocation(n_components=2, random_state=0)

lda.fit(X)

LatentDirichletAllocation(n_components=2, random_state=0)

In [7]:
# Inspect components_

print(lda.components_)  # shape = (n_topics, n_features)

[[1.49838242 0.50114994 2.4966013  0.50114994 1.49838242 1.49838242
  1.49497095 1.49838242 0.50111965 1.49838242 1.49838242 1.49838242
  1.49838242 1.49838242 3.49599211 2.4966013  1.49838242 1.49838242
  0.50111965 1.49838242 0.50114994 0.50111965 0.50111965 0.50111965
  1.49838242 0.50111965 0.50111965 0.50114994]
 [0.50161758 2.49885006 0.5033987  2.49885006 0.50161758 0.50161758
  0.50502905 0.50161758 1.49888035 0.50161758 0.50161758 0.50161758
  0.50161758 0.50161758 0.50400789 0.5033987  0.50161758 0.50161758
  1.49888035 0.50161758 2.49885006 1.49888035 1.49888035 1.49888035
  0.50161758 1.49888035 1.49888035 2.49885006]]


---


lda.components_ : word importance per topic.

    Shape: (n_topics, vocab_size)

    Each value: how much the word contributes to the topic (higher = more important).

    Use .argsort() to get top words per topic.
    

---

In [9]:
# To see top words per topic

import numpy as np

feature_names = vectorizer.get_feature_names_out()
feature_names

array(['artificial', 'bits', 'branch', 'computers', 'data', 'decisions',
       'deep', 'enables', 'exist', 'human', 'identify', 'intelligence',
       'intervention', 'learn', 'learning', 'machine', 'make', 'minimal',
       'multiple', 'patterns', 'quantum', 'qubits', 'simultaneously',
       'states', 'systems', 'traditional', 'unlike', 'use'], dtype=object)

In [10]:
n_top_words = 5

for topic_idx, topic in enumerate(lda.components_):
   
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
   
    print(f"Topic {topic_idx}: {', '.join(top_words)}")


Topic 0: learning, branch, machine, learn, data, decisions, enables, human, identify, intelligence
Topic 1: use, quantum, bits, computers, unlike, exist, multiple, qubits, simultaneously, states


### Getting Topic Distribution per Document 

In [11]:
doc_topics = lda.transform(X)

print(doc_topics)

[[0.96921669 0.03078331]
 [0.91439951 0.08560049]
 [0.03202857 0.96797143]]



Meaning:

Doc 1 → mostly Topic 0

Doc 2 → Topic 0

Doc 3 → Topic 1


### Metrics


Topic Diversity : How distinct the topics are.

Formula:  Unique words across all topics
          --------------------------------
          (total topics × top_k words)


In [12]:

top_words = []

for topic in lda.components_:
    top_words.extend([feature_names[i] for i in topic.argsort()[-5:]])

topic_diversity = len(set(top_words)) / len(top_words)
print("Topic Diversity:", topic_diversity)


Topic Diversity: 1.0



📌 Range: 0 → 1

📌 Closer to 1 = less overlap = better
